# Pythia-1B pilot — Colab
בחר Runtime → Change runtime type → GPU. הרץ את התאים לפי הסדר.
העלה את החבילה `pythia_pilot.zip` שהוכנה בפרויקט.
המשקולות זמניות; תוצאות נשמרות ב־Drive אחרי כל דוגמה.
לא בוצע אימון. בתחילה מריצים רק 24 פרומפטים לבדיקת תקינות.


In [ ]:
from google.colab import files, drive
from pathlib import Path
import os, zipfile, sys, subprocess, shutil

drive.mount('/content/drive')
uploaded = files.upload()
if 'pythia_pilot.zip' not in uploaded:
    raise ValueError('Upload pythia_pilot.zip')
with zipfile.ZipFile('pythia_pilot.zip') as archive:
    for name in archive.namelist():
        target = (Path('/content') / name).resolve()
        if not target.is_relative_to(Path('/content/pythia_pilot')):
            raise ValueError('Unexpected ZIP path: ' + name)
    archive.extractall('/content')
os.chdir('/content/pythia_pilot')
os.environ['PILOT_STORAGE'] = '/content/pythia_pilot/storage'
os.environ['PILOT_RUNS'] = '/content/drive/MyDrive/pythia_pilot_runs'
os.environ['HF_HOME'] = '/content/pythia_pilot/storage/hf'
print('Results:', os.environ['PILOT_RUNS'])


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__); assert torch.cuda.is_available(), "Select a GPU runtime"; print(torch.cuda.get_device_name(0))'], check=True)


In [ ]:
subprocess.run([sys.executable, 'pilot.py', 'download'], check=True)


## הרצה קצרה
אם שינית גרסאות/קוד, בחר שם חדש. חזרה עם אותן הגדרות ממשיכה ריצה חלקית.


In [ ]:
RUN_TAG = 'colab_v1'
subprocess.run([sys.executable, '-u', 'pilot.py', 'run', '--name', RUN_TAG + '_smoke', '--limit-families', '2'], check=True)
print(Path(os.environ['PILOT_RUNS'], RUN_TAG + '_smoke', 'summary.csv').read_text())


## הפיילוט המלא: ארבע הבדיקות, עם ארבע דוגמאות
הרץ לאחר שההרצה הקצרה הסתיימה בהצלחה ובדקת את התוצאות והזמן שלה.


In [ ]:
subprocess.run([sys.executable, '-u', 'pilot.py', 'run', '--name', RUN_TAG + '_4shot'], check=True)


## השוואת 0-shot — אופציונלי, לאחר קריאת התוצאות
אותם מקרים, ללא הדוגמאות שבפרומפט.


In [ ]:
subprocess.run([sys.executable, '-u', 'pilot.py', 'run', '--name', RUN_TAG + '_0shot', '--shots', '0'], check=True)


## הורדת תוצאות למחשב
אפשר להריץ גם אחרי שעצרת ריצה. התוצאות שכבר נכתבו נשמרות ב־Drive.
שמור את ZIP המקומי תחת `final proj/project`.


In [ ]:
archive_path = shutil.make_archive('/content/pythia_results', 'zip', os.environ['PILOT_RUNS'])
files.download(archive_path)
